In [1]:
import pandas as pd
import numpy as np

In [69]:
commodity = pd.read_csv("../02_Silver/commodity.csv")
interest = pd.read_csv("../02_Silver/interest.csv")
makro = pd.read_csv("../02_Silver/gmd.csv")

In [70]:
def add_features(
    df: pd.DataFrame,
    group_cols: list[str],
    date_col: str = "date",
    value_col: str = "value",
    change: str = "pct",  # "pct" | "log" | "diff"
) -> pd.DataFrame:
    """
    Hängt Veränderungen, Volatilitäten und rollierenden 24M-Z-Score pro Gruppe an.

    change:
      "pct"  -> einfache Rendite (P_t / P_{t-n} - 1)
      "log"  -> Log-Rendite ln(P_t / P_{t-n}), nur für Werte > 0
      "diff" -> absolute Differenz P_t - P_{t-n} (bei Zinsen: Prozentpunkte)
    """
    if change not in ("pct", "log", "diff"):
        raise ValueError("change muss 'pct', 'log' oder 'diff' sein")

    out = df.sort_values(group_cols + [date_col]).reset_index(drop=True).copy()
    n_keys = len(group_cols)
    prefix = "chg" if change == "diff" else "ret"

    # Hilfsfunktion: rollierende Statistik pro Gruppe, Index wieder auf Originalindex
    def roll_stat(series: pd.Series, window: int, stat: str) -> pd.Series:
        r = series.groupby([out[c] for c in group_cols]).rolling(window, min_periods=window)
        res = getattr(r, stat)()
        return res.droplevel(list(range(n_keys)))
    # 1.-5. Veränderungen über n Monate
    g = out.groupby(group_cols)[value_col]
    if change == "pct":
        for n in (1, 3, 6, 12, 24):
            out[f"{prefix}_{n}m"] = g.pct_change(periods=n, fill_method=None).replace(
                [np.inf, -np.inf], np.nan
            )
    elif change == "log":
        logv = np.log(out[value_col].where(out[value_col] > 0))
        gl = logv.groupby([out[c] for c in group_cols])
        for n in (1, 3, 6, 12, 24):
            out[f"{prefix}_{n}m"] = gl.diff(n)
    else:  # diff
        for n in (1, 3, 6, 12, 24):
            out[f"{prefix}_{n}m"] = g.diff(n)

    # 6.-10. Volatilität = rollierende Std. der 1M-Veränderungen
    for n in (3, 6, 12, 24):
        out[f"vol_{n}m"] = roll_stat(out[f"{prefix}_1m"], n, "std")

    # 11. Rollierender Z-Score (24M inkl. aktuellem Monat)
    mu = roll_stat(out[value_col], 24, "mean")
    sigma = roll_stat(out[value_col], 24, "std")
    out["zscore_24m"] = (out[value_col] - mu) / sigma.replace(0, np.nan)

    return out


# Rohstoffe
commodity = commodity[['date', 'commodity_name', 'value']]
commodity_features = add_features(commodity, group_cols=["commodity_name"], change="pct")

# Zinsen (country bleibt als Spalte erhalten)
interest = interest[['date', 'iso3','interest_name', 'value']]
rates_features = add_features(interest, group_cols=["iso3", "interest_name"], change="diff")

In [84]:
gdp = makro.query("item_description == 'nGDP_USD' and date.between(2019, 2024)")
top100_iso3 = gdp.groupby("iso3")["value"].mean().nlargest(50).index.tolist()
df_top100 = makro[makro["iso3"].isin(top100_iso3)].copy()

In [72]:
#df_top100.to_excel('makro.xlsx', index=False)
makro = pd.read_excel('makro.xlsx')

In [ ]:
makro = makro.sort_values(by=['iso3', 'date'])
makro['value'] = makro.groupby(['iso3', 'item_description'])['value'].ffill()

In [87]:
makro['value'].isna().sum()

6438

In [88]:
df_top100['value'].isna().sum()

2247

In [89]:
nan_counts = df_top100.groupby('item_description')['value'].agg(nan_count=lambda x: x.isna().sum()).reset_index()
print(nan_counts)

     item_description  nan_count
0                  CA          0
1              CA_GDP          0
2              CA_USD          0
3                 CPI          0
4                 HPI         80
5                  M0        160
6                  M1         64
7                  M2        160
8                  M3         88
9                  M4        384
10               REER          0
11              USDfx          0
12             cbrate         32
13           cgovdebt        200
14       cgovdebt_GDP        200
15            cgovdef         85
16        cgovdef_GDP         85
17            cgovexp         80
18        cgovexp_GDP         80
19            cgovrev         72
20        cgovrev_GDP         72
21            cgovtax         88
22        cgovtax_GDP         88
23               cons          0
24           cons_GDP          0
25           cons_USD          0
26           deflator          0
27            exports          0
28        exports_GDP          0
29        

In [ ]:
makro_pct = makro[makro['change_type'] == 'pct']
makro_diff = makro[makro['change_type'] == 'diff']
makro_cat = makro[makro['change_type'] == "-"]

makro_pct_feat = add_features(makro_pct[['date', 'iso3','item_description', 'value']], group_cols=["iso3", "item_description"], change="pct")
makro_diff_feat = add_features(makro_diff[      ['date', 'iso3','item_description', 'value']], group_cols=["iso3", "item_description"], change="diff")

In [64]:
makro

,countryname,iso3,date,item_description,value,change_type,source,affiliation,unit,EntityType
0,Angola,AGO,2019,nGDP,34500000.0,pct,GlobalMacroData,other,Percent,States
1,Angola,AGO,2020,nGDP,38500000.0,pct,GlobalMacroData,other,Percent,States
2,Angola,AGO,2021,nGDP,53300000.0,pct,GlobalMacroData,other,Percent,States
3,Angola,AGO,2022,nGDP,65600000.0,pct,GlobalMacroData,other,Percent,States
4,Angola,AGO,2023,nGDP,78300000.0,pct,GlobalMacroData,other,Percent,States
...,...,...,...,...,...,...,...,...,...,...
61595,South Africa,ZAF,2022,income_group_code,3.0,-,GlobalMacroData,classification,Rating,States
61596,South Africa,ZAF,2023,income_group_code,3.0,-,GlobalMacroData,classification,Rating,States
61597,South Africa,ZAF,2024,income_group_code,3.0,-,GlobalMacroData,classification,Rating,States
61598,South Africa,ZAF,2025,income_group_code,3.0,-,GlobalMacroData,classification,Rating,States
